MMA  , tiled training performance comparision mlp 
  

In [ ]:

import re
import time
from collections import Counter
import pandas as pd
import cupy as cp
from sentence_transformers import SentenceTransformer
import numpy as np

assert cp.cuda.runtime.getDeviceCount() > 0, "No CUDA GPU detected!"
with cp.cuda.Device(0) as dev:
    props = cp.cuda.runtime.getDeviceProperties(dev.id)
    print(f"[GPU] Using: {props['name'].decode()} (SMs={props['multiProcessorCount']})")

#  Text Cleaning 
def clean_text(s: str) -> str:
    s = str(s)
    s = s.lower()
    s = re.sub(r"[^\w\s$]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

#  Tiled MatMul Kernel 
tiled_kernel_code = r'''
extern "C" __global__
void matmul_tiled(const float* A, const float* B, float* C, int M, int K, int N){
    const int TILE = 16;
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int row = blockIdx.y*TILE + threadIdx.y;
    int col = blockIdx.x*TILE + threadIdx.x;

    float value = 0.0f;
    for(int t=0; t<(K+TILE-1)/TILE; t++){
        int kA = t*TILE + threadIdx.x;
        int kB = t*TILE + threadIdx.y;

        sA[threadIdx.y][threadIdx.x] = (row<M && kA<K) ? A[row*K + kA] : 0.0f;
        sB[threadIdx.y][threadIdx.x] = (kB<K && col<N) ? B[kB*N + col] : 0.0f;
        __syncthreads();

        for(int i=0;i<TILE;i++) value += sA[threadIdx.y][i]*sB[i][threadIdx.x];
        __syncthreads();
    }
    if(row<M && col<N) C[row*N+col] = value;
}
'''
tiled_kernel = cp.RawKernel(tiled_kernel_code, 'matmul_tiled')

def matmul_tiled_fn(A, B):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = cp.zeros((M, N), dtype=cp.float32)
    TILE = 16
    block = (TILE, TILE, 1)
    grid = ((N+TILE-1)//TILE, (M+TILE-1)//TILE, 1)
    tiled_kernel(grid, block, (A, B, C, M, K, N))
    return C

#  MMA / Tensor Core via cuBLAS 
def matmul_mma_fn(A, B):
    A_half = A.astype(cp.float16)
    B_half = B.astype(cp.float16)
    C_half = cp.matmul(A_half, B_half)   
    return C_half.astype(cp.float32)

def softmax(X, axis=1):
    X = X - cp.max(X, axis=axis, keepdims=True)
    exp_X = cp.exp(X)
    return exp_X / cp.sum(exp_X, axis=axis, keepdims=True)

def relu(X):
    return cp.maximum(X, 0)

def xavier_init(in_dim, out_dim):
    limit = cp.sqrt(6.0/(in_dim+out_dim))
    return cp.random.uniform(-limit, limit, size=(in_dim, out_dim)).astype(cp.float32)

#  MLP Classifier 
class MLPClassifier:
    def __init__(self, input_dim, n_classes, hidden_dim=128, matmul_backend='tiled'):
        """
        matmul_backend: 'tiled'  'mma'
        """
        self.matmul_backend = matmul_backend
        self.matmul = matmul_mma_fn if matmul_backend == 'mma' else matmul_tiled_fn

        self.W1 = xavier_init(input_dim, hidden_dim)
        self.b1 = cp.zeros(hidden_dim, dtype=cp.float32)
        self.W2 = xavier_init(hidden_dim, n_classes)
        self.b2 = cp.zeros(n_classes, dtype=cp.float32)

    def forward(self, X):
        Z1 = self.matmul(X, self.W1) + self.b1  
        A1 = relu(Z1)
        logits = self.matmul(A1, self.W2) + self.b2 
        cache = (X, Z1, A1)
        return logits, cache

    def backward_update(self, cache, logits, y, lr=0.1, weight_decay=0.0):
        X, Z1, A1 = cache
        batch = logits.shape[0]
        probs = softmax(logits, axis=1)
        probs[cp.arange(batch), y] -= 1.0
        probs /= batch

        dW2 = self.matmul(A1.T, probs)  
        db2 = cp.sum(probs, axis=0)

        dA1 = self.matmul(probs, self.W2.T)  
        dZ1 = dA1 * (Z1 > 0).astype(cp.float32)  

        dW1 = self.matmul(X.T, dZ1)  
        db1 = cp.sum(dZ1, axis=0)

        if weight_decay > 0:
            dW2 += weight_decay * self.W2
            dW1 += weight_decay * self.W1

        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1

#  Oversample 
def oversample_gpu(X, y):
    counts = Counter(cp.asnumpy(y))
    max_count = max(counts.values())
    X_list = [X]
    y_list = [y]
    for label, count in counts.items():
        if count < max_count:
            idxs = cp.where(y==label)[0]
            reps = cp.random.choice(idxs, size=(max_count-count,), replace=True)
            X_list.append(X[reps])
            y_list.append(y[reps])
    X_new = cp.concatenate(X_list, axis=0)
    y_new = cp.concatenate(y_list, axis=0)
    return X_new, y_new

def train_gpu_sbert(sentences, labels, epochs=40, lr=0.1, hidden_dim=128,
                    batch_size=32, matmul_backend='tiled', weight_decay=1e-4,
                    embedding_model="all-MiniLM-L6-v2"):
    start_time = time.time()

    sentences_clean = [clean_text(s) for s in sentences]
    print("[INFO] Computing Sentence-BERT embeddings on GPU (may take some time)...")
    sbert = SentenceTransformer(embedding_model, device="cuda")
    X_cpu = sbert.encode(sentences_clean, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
    X = cp.asarray(X_cpu, dtype=cp.float32) 
    y = cp.asarray(labels, dtype=cp.int32)

    print("[INFO] Original class counts:", Counter(cp.asnumpy(y)))
    X, y = oversample_gpu(X, y)
    print("[INFO] After oversampling:", Counter(cp.asnumpy(y)))

    n_classes = len(set(cp.asnumpy(y)))
    model = MLPClassifier(X.shape[1], n_classes, hidden_dim=hidden_dim, matmul_backend=matmul_backend)

    N = X.shape[0]
    iter_start = time.time()
    for epoch in range(1, epochs+1):
        perm = cp.random.permutation(N)
        X_shuff = X[perm]
        y_shuff = y[perm]
        epoch_loss = 0.0
        epoch_acc = 0.0
        n_batches = 0

        for i in range(0, N, batch_size):
            xb = X_shuff[i:i+batch_size]
            yb = y_shuff[i:i+batch_size]
            logits, cache = model.forward(xb)
            probs = softmax(logits, axis=1)
            batch_loss = -cp.mean(cp.log(probs[cp.arange(yb.size), yb] + 1e-12))
            epoch_loss += float(batch_loss) * xb.shape[0]
            preds = cp.argmax(logits, axis=1)
            batch_acc = cp.sum(preds == yb).item()
            epoch_acc += batch_acc

            model.backward_update(cache, logits, yb, lr=lr, weight_decay=weight_decay)

            n_batches += 1

        epoch_loss /= N
        epoch_acc = epoch_acc / N * 100.0
        elapsed = time.time() - iter_start
        print(f"Epoch {epoch:02d}/{epochs}  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.2f}%  Time elapsed: {elapsed:.1f}s")
     
    logits_all, _ = model.forward(X)
    preds_all = cp.argmax(logits_all, axis=1)
    total_acc = float(cp.mean((preds_all == y).astype(cp.float32))) * 100.0
    total_time = time.time() - start_time
    print(f"[INFO] Training finished in {total_time:.2f}s")
    print(f"[RESULT] Total Accuracy (on training data after oversample): {total_acc:.2f}%")
    return model, total_acc, total_time

if __name__=="__main__":
    df = pd.read_csv(r"C:\Users\Pavani Akshaya\OneDrive\Desktop\fin_data_1.csv")
    sentences = df["Sentence"].astype(str).tolist()
    lab = df["Sentiment"].astype(str).str.lower().str.strip()
    def map_label(s):
        if s in ("positive", "pos", "1", "p", "true", "t", "positive "):
            return 1
        if s in ("negative", "neg", "0", "n", "false", "f", "negative "):
            return 0
        try:
            v = float(s)
            return 1 if v>0.5 else 0
        except:
            return 1
    labels = [map_label(x) for x in lab]

    print("\n=== TRAINING WITH TILED MATMUL BACKEND ===")
    model_tiled, acc_tiled, time_tiled = train_gpu_sbert(
        sentences, labels,
        epochs=30, lr=0.05, hidden_dim=128, batch_size=32,
        matmul_backend='tiled', weight_decay=1e-4)

    print("\n=== TRAINING WITH MMA/TENSOR-CORE BACKEND ===")
    model_mma, acc_mma, time_mma = train_gpu_sbert(
        sentences, labels,
        epochs=30, lr=0.05, hidden_dim=128, batch_size=32,
        matmul_backend='mma', weight_decay=1e-4)

    print("\n=== COMPARISON ===")
    print(f"Tiled MatMul     - Accuracy: {acc_tiled:.2f}%, Time: {time_tiled:.2f}s")
    print(f"MMA / TensorCore - Accuracy: {acc_mma:.2f}%, Time: {time_mma:.2f}s")


[GPU] Using: NVIDIA GeForce RTX 3060 Laptop GPU (SMs=30)

=== TRAINING WITH TILED MATMUL BACKEND ===
[INFO] Computing Sentence-BERT embeddings on GPU (may take some time)...


Batches: 100%|██████████| 92/92 [00:06<00:00, 13.75it/s]


[INFO] Original class counts: Counter({np.int32(1): 4982, np.int32(0): 860})
[INFO] After oversampling: Counter({np.int32(1): 4982, np.int32(0): 4982})
Epoch 01/30  Loss: 0.6926  Acc: 51.21%  Time elapsed: 1.6s
Epoch 02/30  Loss: 0.6918  Acc: 52.04%  Time elapsed: 0.9s
Epoch 03/30  Loss: 0.6911  Acc: 52.63%  Time elapsed: 0.8s
Epoch 04/30  Loss: 0.6908  Acc: 52.92%  Time elapsed: 0.9s
Epoch 05/30  Loss: 0.6905  Acc: 53.19%  Time elapsed: 0.8s
Epoch 06/30  Loss: 0.6905  Acc: 53.22%  Time elapsed: 0.9s
Epoch 07/30  Loss: 0.6903  Acc: 53.15%  Time elapsed: 0.9s
Epoch 08/30  Loss: 0.6899  Acc: 54.18%  Time elapsed: 0.9s
Epoch 09/30  Loss: 0.6895  Acc: 54.01%  Time elapsed: 0.8s
Epoch 10/30  Loss: 0.6896  Acc: 53.80%  Time elapsed: 1.0s
Epoch 11/30  Loss: 0.6897  Acc: 53.54%  Time elapsed: 0.9s
Epoch 12/30  Loss: 0.6898  Acc: 53.83%  Time elapsed: 1.0s
Epoch 13/30  Loss: 0.6899  Acc: 53.60%  Time elapsed: 0.9s
Epoch 14/30  Loss: 0.6900  Acc: 54.22%  Time elapsed: 1.0s
Epoch 15/30  Loss: 0.6

Batches: 100%|██████████| 92/92 [00:06<00:00, 15.00it/s]


[INFO] Original class counts: Counter({np.int32(1): 4982, np.int32(0): 860})
[INFO] After oversampling: Counter({np.int32(1): 4982, np.int32(0): 4982})
Epoch 01/30  Loss: 0.6484  Acc: 68.82%  Time elapsed: 2.3s
Epoch 02/30  Loss: 0.5330  Acc: 77.16%  Time elapsed: 1.4s
Epoch 03/30  Loss: 0.4575  Acc: 80.22%  Time elapsed: 1.4s
Epoch 04/30  Loss: 0.4243  Acc: 81.83%  Time elapsed: 1.4s
Epoch 05/30  Loss: 0.4067  Acc: 82.40%  Time elapsed: 1.2s
Epoch 06/30  Loss: 0.3962  Acc: 83.04%  Time elapsed: 1.5s
Epoch 07/30  Loss: 0.3874  Acc: 83.61%  Time elapsed: 1.4s
Epoch 08/30  Loss: 0.3816  Acc: 83.85%  Time elapsed: 1.4s
Epoch 09/30  Loss: 0.3765  Acc: 84.24%  Time elapsed: 1.6s
Epoch 10/30  Loss: 0.3721  Acc: 84.60%  Time elapsed: 1.6s
Epoch 11/30  Loss: 0.3669  Acc: 84.73%  Time elapsed: 1.4s
Epoch 12/30  Loss: 0.3637  Acc: 85.15%  Time elapsed: 1.4s
Epoch 13/30  Loss: 0.3607  Acc: 85.15%  Time elapsed: 1.7s
Epoch 14/30  Loss: 0.3576  Acc: 85.12%  Time elapsed: 1.6s
Epoch 15/30  Loss: 0.3